# Monte Carlo estimation and introductory Metropolis sampling

## Session 4: Introduction to Monte Carlo simulation

Use the Week 2 page in ELO (Session 4 section) and exercises S4.1–S4.5.
Consult the [notation glossary](NotationGlossary.pdf) as needed;
it is a reference, not an additional assignment.
Work through sections A–D, then **stop before Later sessions**.
These tasks cover the existing simulation exercises S1–S2 once.

**Weekly timing:** Sessions 3 and 4 take place on the same day, with a
15-minute rest break. No homework or setup is required during that gap.
Complete this follow-up after both sessions and before Session 5 the next week.

The follow-up is 30 minutes reading and 45 minutes practice: 10 worksheet,
25 notebook, 10 discussion and your own explanation. No new package is needed.
We estimate an integral with a known reference, compare complete independent
estimates, and distinguish observation spread from estimation error.

In [ ]:
import sys
import math
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

print('Python', sys.version.split()[0])
print('NumPy', np.__version__)
print('Matplotlib', matplotlib.__version__)

## A. An integral as an expectation

Work on a probability space $(\Omega,\mathcal F,\Pr)$, where $\Omega$
is the sample space, $\mathcal F$ its sigma-algebra of events, and $\Pr$
its probability measure. Let $a,b\in\mathbb R$ with $a<b$.
Let $g:[a,b]\to\mathbb R$ be Borel
measurable and $\int_a^b|g(x)|\,dx<\infty$. For $U\sim\mathrm{Uniform}[a,b]$,
$$
I=\int_a^b g(x)\,dx=(b-a)\,\mathbb E[g(U)].
$$
Define $Y=(b-a)g(U)$, so $\mathbb E|Y|<\infty$ and $\mathbb E[Y]=I$.
The uniform density is $1/(b-a)$ on $[a,b]$, so
$\mathbb E[g(U)]=(b-a)^{-1}\int_a^b g(x)\,dx$. Multiplying by the interval
width recovers the integral. Here $dx$ denotes integration with respect to
Lebesgue measure on the real line. For $g(x)=x^2$ on $[0,2]$, the integral
is $8/3$, while $\mathbb E[g(U)]=4/3$ (S4.1). The contribution is
$Y=2U^2$, whose expectation is $8/3$.

Our main example is $g:[0,1]\to\mathbb R$, $g(u)=\exp(-u^2)$, with
$U\sim\mathrm{Uniform}[0,1]$. The width equals one.
The error function $\operatorname{erf}:\mathbb R\to\mathbb R$ is defined by
$\operatorname{erf}(t)=2\pi^{-1/2}\int_0^t e^{-v^2}\,dv$ for $t\in\mathbb R$,
where $\pi$ is the circle constant. It supplies the reference
$$
I=\frac{\sqrt{\pi}}{2}\operatorname{erf}(1).
$$
Python evaluates this known expression numerically. The estimator does not
use the reference. This simple integral is a validation example, not evidence
that Monte Carlo is the most efficient method for one-dimensional integration.

In [ ]:
def integrand(x):
    return np.exp(-np.asarray(x)**2)

reference_I = math.sqrt(math.pi) * math.erf(1.0) / 2
reference_second_moment = math.sqrt(math.pi) * math.erf(math.sqrt(2)) / (2*math.sqrt(2))
reference_variance = reference_second_moment - reference_I**2
reference_sd = math.sqrt(reference_variance)
print(f'Reference integral: {reference_I:.12f}')
print(f'Model observation SD: {reference_sd:.8f}')

grid = np.linspace(0, 1, 301)
fig_integrand, ax = plt.subplots(figsize=(6.5, 3.5))
ax.plot(grid, integrand(grid), color='#246a91', label='g(u) = exp(-u²)')
ax.fill_between(grid, 0, integrand(grid), color='#246a91', alpha=.15)
ax.set(xlabel='u', ylabel='g(u)', xlim=(0, 1), ylim=(0, 1.08))
ax.legend()
fig_integrand.tight_layout()
plt.show()

## B. One complete estimate

For $n\in\mathbb N=\{1,2,\ldots\}$, take iid
$U_1,\ldots,U_n\sim\mathrm{Uniform}[a,b]$ and set $Y_i=(b-a)g(U_i)$
for $i=1,\ldots,n$. Under the integrability assumption in A,
$$
\widehat I_n=\frac1n\sum_{i=1}^n Y_i,\qquad
\mathbb E[\widehat I_n]=I.
$$
Thus the estimator is unbiased: its expectation equals the target integral.
Assume additionally $\int_a^b g(x)^2\,dx<\infty$,
as holds for our bounded integrand. Write $\sigma_Y^2=\operatorname{Var}(Y_1)$
and $\sigma_Y=\sqrt{\sigma_Y^2}$. Independence and equal variances give
$$\operatorname{Var}(\widehat I_n)=\frac{\sigma_Y^2}{n},\qquad
\operatorname{SE}(\widehat I_n)=\frac{\sigma_Y}{\sqrt n}.$$
Unbiasedness itself needs no independence when all means equal $I$.

For a realized run $\omega\in\Omega$ in this probability space,
write $u_i=U_i(\omega)$, $y_i=Y_i(\omega)$ and
$\widehat I_n(\omega)=\overline y_n=n^{-1}\sum_{i=1}^ny_i$.
For $n\geq2$, the observed sample SD and estimated SE are
$$s_{Y,n}=\sqrt{\frac1{n-1}\sum_{i=1}^n(y_i-\overline y_n)^2},\qquad
\widehat{\operatorname{SE}}_n=\frac{s_{Y,n}}{\sqrt n}.$$
`estimate`, `observation_sd` and `estimated_se` below are these observed values.
The realized signed error is $\widehat I_n(\omega)-I$; the absolute error is
$|\widehat I_n(\omega)-I|$. We approximate them numerically using the reference.
The SE describes the distribution of the random estimator; it is neither the
realized error nor a guaranteed error bound. NumPy's generator approximates
the iid model using pseudorandom draws.

**Predict (S4.2):** Why does the function average the transformed values rather
than evaluating the integrand at the average input? Why does it require n >= 2
when reporting a sample-based SE?

In [ ]:
def mc_uniform(g, a, b, n, rng):
    '''Integrate a vectorized scalar integrand using n iid uniform draws.'''
    if not (np.isfinite(a) and np.isfinite(b) and a < b):
        raise ValueError('use finite bounds with a < b')
    if not isinstance(n, (int, np.integer)) or isinstance(n, bool) or n < 2:
        raise ValueError('n must be an integer >= 2 to estimate the SE')
    draws = rng.uniform(a, b, size=n)
    values = np.asarray(g(draws), dtype=float)
    if values.shape != draws.shape or not np.all(np.isfinite(values)):
        raise ValueError('g must return one finite value per input')
    contributions = (b-a) * values
    observation_sd = contributions.std(ddof=1)
    return dict(estimate=float(contributions.mean()),
                observation_sd=float(observation_sd),
                estimated_se=float(observation_sd / np.sqrt(n)),
                draws=draws, values=values, contributions=contributions,
                evaluations=n)

single_n = 1600
single_seed = 84
single = mc_uniform(integrand, 0., 1., single_n, np.random.default_rng(single_seed))
print(f"Estimate: {single['estimate']:.8f}")
print(f"Signed error against reference: {single['estimate'] - reference_I:+.8f}")
print(f"Observation sample SD: {single['observation_sd']:.8f}")
print(f"Estimated SE: {single['estimated_se']:.8f}")
print(f"Model SE: {reference_sd / np.sqrt(single_n):.8f}")
print('Integrand evaluations:', single['evaluations'])
assert np.all((single['draws'] >= 0) & (single['draws'] <= 1))
assert math.exp(-1) <= single['estimate'] <= 1

Each contribution costs one integrand evaluation, even though NumPy evaluates
the array in one Python call. The plot grid is separate illustration work.
The estimator requires no additional random vertical coordinate.

**Prediction:** Will the absolute error of a running estimate decrease at
every new sample, as the retained best objective value did in Session 3?
For $m\in\{1,\ldots,1600\}$, the running estimate is
$\overline y_m=m^{-1}\sum_{i=1}^m y_i$. The next cell uses prefixes of the
same run; neighbouring estimates share observations and are dependent.

In the left plot, each observation is one transformed contribution
$y_i=e^{-u_i^2}$, not a whole-run estimate. For a histogram bin of width
$h>0$ containing $c$ of the 1600 observations, the plotted density height
is $c/(1600h)$. Its area is the observed fraction $c/1600$. The dashed line
marks the reference mean $I$. In the right plot, each point averages the
first $m$ contributions; its horizontal position is $m$. These plots answer
different questions about the same run.

In [ ]:
running_n = np.arange(1, single_n+1)
running_estimate = np.cumsum(single['contributions']) / running_n
running_error = np.abs(running_estimate - reference_I)
print('At least one increase in running absolute error:', bool(np.any(np.diff(running_error) > 0)))

fig_single, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].hist(single['contributions'], bins=20, density=True, color='#246a91', alpha=.7)
axes[0].axvline(reference_I, color='black', linestyle='--', label='Reference mean')
axes[0].set(xlabel='Observed contribution y = exp(-u²)', ylabel='Density',
            title='1600 individual contributions')
axes[0].legend(fontsize=8)
axes[1].plot(running_n, running_estimate, color='#246a91', linewidth=.9)
axes[1].axhline(reference_I, color='black', linestyle='--', label='Reference integral')
axes[1].set(xlabel='Samples in one run', ylabel='Running estimate', title='One sequence of averages')
axes[1].legend(fontsize=8)
fig_single.tight_layout()
plt.show()

## C. Repeated complete estimates

**Predict (S4.3 / S1):** For a fixed contribution distribution with
$0<\sigma_Y^2<\infty$, multiplying $n$ by four halves the model SE. Does it
halve every realized absolute error? Does it halve the spread of individual $Y_i$?

Use 200 repetitions for each of n = 100, 400, 1600. Each complete estimate
gets a distinct child stream from root seed 4040. The model assumes independence
within a run and across complete repetitions. Record n and the repetition
count separately. The results array has shape `(3, 200)`: rows use
$n=100,400,1600$, respectively, and each column identifies a complete run
within a row. For example, entry `[1, 0]` is the first estimate using 400
draws. The array stores estimates, not the underlying draws. Thus
`.std(axis=1, ddof=1)` computes one sample SD across 200 estimates for each
sample size; `.mean(axis=1)` computes their row means.
Distinct pseudorandom streams approximate the independent model; their labels
alone do not prove probabilistic independence.

For each fixed $n$, let $\widehat I_n^{(1)},\ldots,\widehat I_n^{(R)}$ be
$R=200$ independent copies of $\widehat I_n$. For observed values
$t_r=\widehat I_n^{(r)}(\omega)$, $r=1,\ldots,R$, the empirical SD is
$$\overline t_R=\frac1R\sum_{r=1}^R t_r,\qquad
s_{\mathrm{rep}}=\sqrt{\frac1{R-1}\sum_{r=1}^R(t_r-\overline t_R)^2}.$$
It estimates the SE of **one** $n$-sample estimator. For contrast, define
the random average $A_{n,R}=R^{-1}\sum_{r=1}^R\widehat I_n^{(r)}$.
Its observed value is $\overline t_R$, and under the independent-copy model
$\operatorname{SE}(A_{n,R})=\sigma_Y/\sqrt{nR}$.
Dividing $s_{\mathrm{rep}}$ by $\sqrt R$ estimates this latter SE.

For $n=400$, the known $\sigma_Y\approx0.20099184$ gives
$\operatorname{SE}(\widehat I_{400})\approx0.01004959$.
With $R=200$, $\operatorname{SE}(A_{400,200})\approx0.00071061$.
The second estimator averages 80,000 contributions in total; it is a
different estimator from one 400-draw run.

The first plot compares all complete estimates with the known answer.
The second compares their empirical SD with the predicted SE. An empirical
SD from 200 repetitions still varies. It is not expected to equal the theory exactly.

In [ ]:
def repeat_integral(sample_sizes, repetitions, root_seed):
    if repetitions < 2:
        raise ValueError('at least two repetitions are needed for empirical SD')
    children = np.random.SeedSequence(root_seed).spawn(len(sample_sizes)*repetitions)
    estimates = np.empty((len(sample_sizes), repetitions))
    observation_sd = np.empty_like(estimates)
    estimated_se = np.empty_like(estimates)
    for j, n in enumerate(sample_sizes):
        for k in range(repetitions):
            rng = np.random.default_rng(children[j*repetitions+k])
            run = mc_uniform(integrand, 0., 1., int(n), rng)
            estimates[j, k] = run['estimate']
            observation_sd[j, k] = run['observation_sd']
            estimated_se[j, k] = run['estimated_se']
    return dict(estimates=estimates, observation_sd=observation_sd, estimated_se=estimated_se)

In [ ]:
sample_sizes = np.array([100, 400, 1600])  # Change only 400 to 900 for section D.
repetition_count = 200
comparison_seed = 4040
comparison = repeat_integral(sample_sizes, repetition_count, comparison_seed)
estimate_errors = comparison['estimates'] - reference_I
empirical_sd = comparison['estimates'].std(axis=1, ddof=1)
mean_estimated_se = comparison['estimated_se'].mean(axis=1)
mean_observation_sd = comparison['observation_sd'].mean(axis=1)
theoretical_se = reference_sd / np.sqrt(sample_sizes)
print('n | mean estimate | mean observation SD | mean estimated SE | SD of estimates | model SE')
for j, n in enumerate(sample_sizes):
    print(f'{n:4d} | {comparison["estimates"][j].mean():.8f} | {mean_observation_sd[j]:.6f} | '
          f'{mean_estimated_se[j]:.6f} | {empirical_sd[j]:.6f} | {theoretical_se[j]:.6f}')
print('Total integrand evaluations in the repetition experiment:', int(repetition_count * sample_sizes.sum()))

fig_repetitions, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].boxplot(estimate_errors.T, tick_labels=sample_sizes, showfliers=True)
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set(xlabel='Samples per estimate n', ylabel='Estimate minus reference',
            title=f'{repetition_count} complete estimates per n')
positions = np.arange(len(sample_sizes))
axes[1].bar(positions-.18, empirical_sd, width=.36, color='#246a91', label='Empirical SD of estimates')
axes[1].bar(positions+.18, theoretical_se, width=.36, color='#b65e2f', label='Model SE')
axes[1].set(xticks=positions, xticklabels=sample_sizes, xlabel='Samples per estimate n',
            ylabel='Spread of complete estimates', title='Observed spread and model prediction')
axes[1].legend(fontsize=8)
fig_repetitions.tight_layout()
plt.show()

Box centers mark medians; boxes extend from the first quartile $Q_1$ to the
third quartile $Q_3$, using linear interpolation as defined in the glossary.
The interquartile range is $\mathrm{IQR}=Q_3-Q_1$. Whiskers reach the smallest
and largest observations in $[Q_1-1.5\mathrm{IQR},Q_3+1.5\mathrm{IQR}]$;
dots show the remaining results. All repetitions contribute to the summaries.
These boxes describe variability of complete estimates, not confidence intervals.

The mean observation SD averages the $R$ within-run sample SDs. It is not
generally an unbiased estimate of $\sigma_Y$, although it is near that value
here. The model SE decreases with $n$. Doubling $n$ does not halve it;
quadrupling $n$ does, for the same positive finite variance.
Formal interval coverage and detailed convergence analysis return later.

## D. One controlled change and your explanation

Within the 25-minute notebook allocation:

1. Preserve the baseline table or a short note of its middle row.
2. Predict the new SE if the middle sample size changes from 400 to 900.
3. Change only 400 to 900 in sample_sizes, keeping 200 repetitions and root
   seed 4040, then rerun the comparison cell. No parameter sweep is required.
4. Compare the changed middle row with the baseline. The model SE becomes
   two thirds of its previous value. The other groups use the same child
   streams and sample sizes, so their results stay the same in this environment.
   The middle groups reuse stream prefixes: the baseline and changed results
   are dependent across settings, although repetitions within each setting
   approximate independent copies.
5. Explain why a larger n changes uncertainty in an estimate but does not
   concentrate the distribution of individual contributions at the integral.

The theoretical reduction describes variability, not a guaranteed reduction
of each observed error. If setup is blocked, use the notes table and the
theoretical ratio; distinguish supplied evidence from predictions.

**Stop here for Session 4.** In the ten-minute discussion block, compare one
claim with a peer and write your own three-sentence explanation: the expectation
being estimated, observation SD versus estimator SE, and one result with a limitation.
The course collaboration link announced in ELO lets you
share this existing explanation; uploading is optional and ungraded.

Session 3 retained a best point; this session averages all transformed draws.
A possible simulation project compares the accuracy and cost of estimators.
The individual project is chosen after both introductory fields at the end
of week 4. This example adds no project work.

References for the supplied computation:
[Python error function](https://docs.python.org/3/library/math.html#math.erf),
[NumPy sample SD](https://numpy.org/doc/stable/reference/generated/numpy.std.html),
and [child streams](https://numpy.org/doc/stable/reference/random/parallel.html).
The formulas in this session are derived from expectation and iid variance.

# Later sessions: independent-estimate recap and Metropolis sampling

**Stop here for Session 4.** S1–S2 are already covered by the opening Session 4
route. The remaining cells provide a recap and an introductory Metropolis
demonstration for later sessions; they add no Session 4 homework.

Use S3–S6 when assigned. Detailed stationarity and convergence analysis return
in the later simulation block. Identify supplied code when reusing it.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

exact_integral = math.sqrt(math.pi) * math.erf(1.0) / 2.0

def monte_carlo_integral(n, rng):
    if not isinstance(n, (int, np.integer)) or n < 2:
        raise ValueError("n must be an integer >= 2 for a sample-based SE.")
    values = np.exp(-rng.uniform(size=n)**2)
    return values.mean(), values.std(ddof=1) / np.sqrt(n)

estimate, standard_error = monte_carlo_integral(4000, np.random.default_rng(84))
print(f'Exact integral: {exact_integral:.6f}')
print(f'Estimate: {estimate:.6f}; estimated iid standard error: {standard_error:.6f}')

## Independent estimates
This recap uses iid $U_i\sim\mathrm{Uniform}[0,1]$ and $Y_i=e^{-U_i^2}$,
target $I=\int_0^1e^{-u^2}\,du$, and random estimator
$\widehat I_n=n^{-1}\sum_{i=1}^nY_i$. For $n\geq2$, observed SE is
$s_{Y,n}/\sqrt n$, with sample SD denominator $n-1$. Each plotted error is
$\widehat I_n^{(r)}(\omega)-I$ for one of $R=60$ complete repetitions.
The model assumes independent repetitions, approximated by separate child streams.
Predict how their spread changes with $n$; individual realized errors need not
decrease. This recaps Session 4, not a second required assignment.

In [ ]:
sizes = [100, 400, 1600, 6400]
repetitions = 60
seeds = np.random.SeedSequence(84).spawn(len(sizes) * repetitions)
errors = []
for j, n in enumerate(sizes):
    errors.append([monte_carlo_integral(n, np.random.default_rng(seeds[j*repetitions+k]))[0] - exact_integral for k in range(repetitions)])
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.boxplot(errors, tick_labels=sizes)
ax.axhline(0, color='black', linestyle='--')
ax.set(xlabel='Independent draws per estimate', ylabel='Estimate minus exact value', title='Repeated Monte Carlo integration experiments')
fig.tight_layout()
plt.show()

## A target known up to a constant
Let $\pi=\mathcal N(\mu,\sigma^2)$ be the target law on $\mathbb R$,
with $\mu=1$, $\sigma=1.5$. Its density is
$p(x)=\exp[-(x-\mu)^2/(2\sigma^2)]/(\sigma\sqrt{2\pi})$;
the $\pi$ inside the square root is the circle constant. The log target
$\ell(x)=-(x-\mu)^2/(2\sigma^2)$ differs from $\log p(x)$ by a constant.
Known moments and density are validation references. In harder problems,
direct independent draws may be unavailable.

At current $x$, use $y=x+sZ$, where $s>0$, $Z\sim\mathcal N(0,1)$ is fresh,
and the proposal density $q(y\mid x)$ is symmetric. Accept with probability
$\alpha(x,y)=\min\{1,e^{\ell(y)-\ell(x)}\}$ using an independent uniform draw.
Record the current state again after rejection. Predict a very small scale's effect.

**Index convention:** this introductory helper returns `draws` stored states
including $X_0$: with `draws=5000`, it makes 4999 transitions and stores
$X_0,\ldots,X_{4999}$. Slicing `[800:]` retains $X_{800},\ldots,X_{4999}$,
4200 states. In contrast, SIM-02 and later helpers specify post-transition
states only; adapt indexing explicitly. The displayed pooled mean/SD are
descriptive summaries, not proof of stationarity or an iid standard error.

In [ ]:
target_mean, target_sd = 1.0, 1.5

def log_target(x):
    return -0.5 * ((x - target_mean) / target_sd)**2

def metropolis(log_density, start, draws, proposal_scale, rng):
    if not isinstance(draws, (int, np.integer)) or draws < 2 or proposal_scale <= 0:
        raise ValueError("Use at least two stored states and positive proposal scale.")
    chain = np.empty(draws)
    current = float(start)
    current_log_density = log_density(current)
    accepted = 0
    chain[0] = current
    for i in range(1, draws):
        proposal = current + rng.normal(scale=proposal_scale)
        proposal_log_density = log_density(proposal)
        log_ratio = proposal_log_density - current_log_density
        if log_ratio >= 0 or rng.random() < np.exp(log_ratio):
            current, current_log_density = proposal, proposal_log_density
            accepted += 1
        chain[i] = current  # Retain repeated states after rejection.
    return chain, accepted / (draws - 1)

In [ ]:
draws = 5000
warmup = 800  # An illustrative choice, not a guarantee of convergence.
starts = [-8.0, -2.0, 4.0, 10.0]
scales = [0.05, 1.5, 8.0]
seeds = np.random.SeedSequence(2026).spawn(len(starts) * len(scales))
chains_by_scale = {}
for j, scale in enumerate(scales):
    chains, rates = [], []
    for k, start in enumerate(starts):
        chain, rate = metropolis(log_target, start, draws, scale, np.random.default_rng(seeds[j*len(starts)+k]))
        chains.append(chain)
        rates.append(rate)
    chains_by_scale[scale] = np.asarray(chains)
    retained = np.asarray(chains)[:, warmup:]
    print(f'Scale={scale}: acceptance={np.mean(rates):.3f}, mean={retained.mean():.3f}, SD={retained.std(ddof=1):.3f}')
print(f'Target mean={target_mean}, target SD={target_sd}')

In [ ]:
fig, axes = plt.subplots(len(scales), 2, figsize=(10, 8))
grid = np.linspace(-5, 7, 400)
density = np.exp(-0.5*((grid-target_mean)/target_sd)**2) / (target_sd*np.sqrt(2*np.pi))
for row, scale in enumerate(scales):
    chains = chains_by_scale[scale]
    for chain in chains:
        axes[row, 0].plot(chain, linewidth=0.6, alpha=0.75)
    axes[row, 0].axhline(target_mean, color='black', linestyle='--')
    axes[row, 0].set(title=f'Proposal scale {scale}', xlabel='Iteration', ylabel='State')
    axes[row, 1].hist(chains[:, warmup:].ravel(), bins=40, density=True, color='#36789a', alpha=0.65)
    axes[row, 1].plot(grid, density, color='black', label='Exact density')
    axes[row, 1].set(xlabel='State', ylabel='Density')
    axes[row, 1].legend()
fig.tight_layout()
plt.show()

## Your investigation
1. Recall S1–S2 from Session 4; use the independent-estimate recap if needed.
2. Trace one acceptance calculation by hand and answer S3–S4.
3. Inspect the traces separately before interpreting pooled moments or histograms. Which initializations are still visible?
4. Complete S5: changing `warmup` alone does not repair a slowly exploring chain. Do not use sample SD divided by square root of the raw chain length as an iid standard error for these correlated draws.
5. Use S6 to discuss an unexpected result and revise your own explanation.

Later sessions introduce autocorrelation, effective sample size, Monte Carlo error for dependent draws, and multiple-chain diagnostics. Here the known target provides an initial validation reference.